# Hallucination and grounding

Exercise notebook for the lesson **[Hallucination and grounding](https://ramkumarmd.com/medical-ai/generative-ai-llms/hallucination-and-grounding)** in the *Generative AI & LLMs* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** In clinical text, a hallucination is content that is not supported by the source, such as a medicine the patient was never prescribed; an omission is something important that was left out. Both occur even in good systems: one large evaluation of AI-generated clinical notes found hallucinations in 1.47% of sentences, almost half of them potentially harmful. Grounding the model in specific source documents, requiring traceable citations and checking outputs against the source reduce the risk, but a clinician must still review what matters.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Measuring errors with clinician review

Summarising a clinician review of four synthetic AI-generated notes.

In [ ]:
reviews = [
    {"note_sentences": 14, "hallucinations": [], "omissions": ["allergy"]},
    {"note_sentences": 11, "hallucinations": [("dose changed", "major")], "omissions": []},
    {"note_sentences": 16, "hallucinations": [("extra symptom", "minor")],
     "omissions": ["follow-up date", "smoking status"]},
    {"note_sentences": 12, "hallucinations": [], "omissions": []},
]
sentences = sum(r["note_sentences"] for r in reviews)
halluc = [h for r in reviews for h in r["hallucinations"]]
omissions = sum(len(r["omissions"]) for r in reviews)
major = sum(1 for _, severity in halluc if severity == "major")
print(f"{sentences} sentences reviewed")
print(f"Hallucination rate {len(halluc) / sentences:.1%} ({major} major)")
print(f"Omissions: {omissions} across {len(reviews)} notes")
print(f"Notes with at least one error: "
      f"{sum(1 for r in reviews if r['hallucinations'] or r['omissions'])} of {len(reviews)}")

# 53 sentences reviewed
# Hallucination rate 3.8% (1 major)
# Omissions: 3 across 4 notes
# Notes with at least one error: 3 of 4

### A simple automated check, and its limits

Flag summary sentences containing numbers or medicines that are not in the synthetic source.

In [ ]:
import re

source = """Synthetic consultation. Patient reports tiredness for three weeks.
On metformin 1 g twice daily. HbA1c last month 8.4%. Blood pressure today 138/86.
No chest pain. Plan: add a second medicine after discussing options; recheck HbA1c in 3 months."""

summary_sentences = [
    "Three weeks of tiredness.",
    "Takes metformin 1 g twice daily.",
    "HbA1c 8.4% last month; blood pressure 138/86.",
    "Started on gliclazide 80 mg.",          # not in the source
    "Reports chest pain on exertion.",       # contradicts the source
    "Recheck HbA1c in 6 months.",            # wrong interval
]

MEDICINES = {"metformin", "gliclazide", "insulin", "sitagliptin", "empagliflozin"}
src = source.lower()
src_numbers = set(re.findall(r"\d+(?:\.\d+)?", src))

def unsupported_details(sentence):
    s = sentence.lower()
    problems = [f"number {n}" for n in re.findall(r"\d+(?:\.\d+)?", s) if n not in src_numbers]
    problems += [f"medicine {m}" for m in MEDICINES if m in s and m not in src]
    return problems

for sentence in summary_sentences:
    problems = unsupported_details(sentence)
    print(("FLAG  " if problems else "ok    ") + sentence, problems or "")

# ok    Three weeks of tiredness.
# ok    Takes metformin 1 g twice daily.
# ok    HbA1c 8.4% last month; blood pressure 138/86.
# FLAG  Started on gliclazide 80 mg. ['number 80', 'medicine gliclazide']
# ok    Reports chest pain on exertion.
# FLAG  Recheck HbA1c in 6 months. ['number 6']

## Exercise

**Goal:** Measure and reduce hallucinations and omissions in synthetic AI-generated clinic notes.

**Data:** Ten synthetic consultation transcripts with AI-style draft notes, written for the notebook (Created for this course; no real patient data)

### Step 1

Review two draft notes sentence by sentence against their transcripts and label hallucinations, contradictions and omissions with severity.

In [ ]:
# Your code here


### Step 2

Compute per-sentence and per-note error rates for all ten notes using the labels provided.

In [ ]:
# Your code here


### Step 3

Run the number-and-medicine checker and calculate its sensitivity for the labelled hallucinations.

In [ ]:
# Your code here


### Step 4

Add a simple negation rule (for example 'no', 'denies', 'without' before a symptom) and measure the improvement.

In [ ]:
# Your code here


### Step 5

Write a one-page review protocol proportionate to how these notes would be used.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/generative-ai-llms/hallucination-and-grounding](https://ramkumarmd.com/medical-ai/generative-ai-llms/hallucination-and-grounding). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.